In [1]:
import scarf
from scarf.agent import (
    BiologicalContext,
    BiologicalInterpretationAgent,
    DataEnrichmentAgent,
    DataEnrichmentContext,
    ExperimentalContextAgent,
    ParameterCandidate,
    ParameterTuningAgent,
)

scarf.configure_output(level="WARNING", progress=False)

dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    "tenx_5K_pbmc_rnaseq",
    destination="scarf_datasets",
    zarr=True,
)
ds = scarf.DataStore(
    f"{dataset}/data.zarr",
    default_assay="RNA",
    nthreads=2,
)

{
    "active_cells": int(ds.cells.fetch_all("I").sum()),
    "total_cells": ds.cells.N,
    "assays": ds.assay_names,
}

{'active_cells': 3940, 'total_cells': 5025, 'assays': ['RNA']}

In [2]:
from pydantic_ai.messages import (
    ModelMessage,
    ModelResponse,
    ToolCallPart,
    ToolReturnPart,
)
from pydantic_ai.models.function import AgentInfo, FunctionModel

from scarf.agent.biological_interpretation import (
    ClusterCompositionEvidence,
    ClusterMarkerBatchEvidence,
)
from scarf.agent.data_enrichment import AssayFeatureInspectionBatch
from scarf.agent.experimental_context import CovariateEvidence


def _tool_returns(messages: list[ModelMessage]) -> list[ToolReturnPart]:
    return [
        part
        for message in messages
        for part in message.parts
        if isinstance(part, ToolReturnPart)
    ]


def _tool_call(name: str, args: dict | None = None) -> ModelResponse:
    return ModelResponse(parts=[ToolCallPart(tool_name=name, args=args or {})])


def _structured_output(info: AgentInfo, payload: dict) -> ModelResponse:
    return _tool_call(info.output_tools[0].name, payload)


async def _enrichment_reply(
    messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    returns = _tool_returns(messages)
    if not returns:
        return _tool_call("inspect_assay_features_batch")

    batch = AssayFeatureInspectionBatch.model_validate(returns[-1].content)
    inspection = batch.inspections[0]
    species_observed = inspection.species != "unknown"
    species = inspection.species if species_observed else "homo_sapiens"
    evidence_ids = list(inspection.evidenceIds)
    if not species_observed:
        evidence_ids.append("context:organism")
    policy = {
        "assay": inspection.assay,
        "species": species,
        "speciesConfidence": "high" if species_observed else "medium",
        "speciesRationale": (
            inspection.speciesReason
            or "The inspected features and caller context support this species."
        ),
        "excludeFamilies": [
            family.family
            for family in inspection.families
            if family.count > 0 and family.defaultExclude is True
        ],
        "protectFamilies": [
            family.family
            for family in inspection.families
            if family.count > 0 and family.defaultExclude is False
        ],
        "rationale": "Exclude observed technical families and preserve protected ones.",
        "evidenceIds": evidence_ids,
    }
    return _structured_output(info, {"status": "done", "policies": [policy]})


async def _experimental_reply(
    messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    returns = _tool_returns(messages)
    if not returns:
        return _tool_call("inspect_cell_covariates")
    if len(returns) == 1:
        return _tool_call(
            "analyze_experimental_design",
            {
                "column_domains": {},
                "coefficients_of_interest": [],
                "units_of_inference": {},
                "batch_columns": [],
            },
        )

    design = CovariateEvidence.model_validate(returns[-1].content)
    evidence_id = design.evidenceIds[0]
    return _structured_output(
        info,
        {
            "batchCorrection": {
                "action": "skip",
                "rationale": (
                    "No explicit technical batch or biological contrast was supplied."
                ),
                "evidenceIds": [evidence_id],
            },
            "rationale": "Continue with an uncorrected baseline.",
            "evidenceIds": [evidence_id],
        },
    )


async def _tuning_reply(
    _messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    return _structured_output(
        info,
        {
            "status": "done",
            "recommendedCandidateId": "baseline",
            "confidence": "medium",
            "rationale": "The single authorized baseline completed successfully.",
            "evidenceIds": ["candidate:baseline:clusters"],
            "stopReason": "The authorized candidate was evaluated.",
        },
    )


async def _biology_reply(
    messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    returns = _tool_returns(messages)
    if not returns:
        return _tool_call("inspect_cluster_composition")
    if len(returns) == 1:
        composition = ClusterCompositionEvidence.model_validate(returns[-1].content)
        cluster_id = sorted(
            composition.clusterCounts,
            key=lambda value: (-composition.clusterCounts[value], value),
        )[0]
        return _tool_call(
            "inspect_cluster_markers_batch",
            {"cluster_ids": [cluster_id]},
        )

    marker_batch = ClusterMarkerBatchEvidence.model_validate(returns[-1].content)
    marker = marker_batch.clusters[0]
    if not marker.evidenceId:
        return _structured_output(
            info,
            {
                "status": "needsInput",
                "needsInput": {
                    "question": "No markers passed the bounded search thresholds.",
                    "requiredInputs": ["markerArtifact"],
                },
                "limitations": marker.warnings,
                "stopReason": "Marker evidence was unavailable.",
            },
        )
    names = [item.featureName or item.featureId for item in marker.markers[:3]]
    return _structured_output(
        info,
        {
            "status": "done",
            "clusterInterpretations": [
                {
                    "clusterId": marker.clusterId,
                    "proposedIdentity": "unresolved marker-defined cluster",
                    "identityIsHypothesis": True,
                    "confidence": "low",
                    "rationale": f"Top returned marker features: {', '.join(names)}.",
                    "evidenceIds": [marker.evidenceId],
                }
            ],
            "evidenceIds": [marker.evidenceId],
            "limitations": [
                "The scripted documentation model does not assign cell identities."
            ],
            "stopReason": "One bounded cluster was reviewed.",
        },
    )

In [3]:
enrichment = DataEnrichmentAgent(FunctionModel(_enrichment_reply)).run(
    ds,
    context=DataEnrichmentContext(
        studyContext=(
            "10x 5K PBMC RNA-seq from peripheral blood of a healthy human donor."
        ),
        organismHint="human",
        tissueReferences=["peripheral blood"],
        cellTypeReferences=["T cell", "B cell", "NK cell", "monocyte"],
        experimentalDetails=["10x 3 prime RNA-seq", "single donor"],
    ),
    assays=["RNA"],
)

policy = enrichment.policies[0]
{
    "status": enrichment.status,
    "species": policy.species,
    "exclude_families": policy.excludeFamilies,
    "protect_families": policy.protectFamilies,
    "tool_calls": [call.name for call in enrichment.toolCalls],
}

{'status': 'done',
 'species': 'homo_sapiens',
 'exclude_families': ['ribosomal', 'histone'],
 'protect_families': ['cellCycle'],
 'tool_calls': ['inspect_assay_features_batch']}

In [4]:
run = ds.pipeline.open(label="docs_default")
normalized = run["normalized"]
hvg_ref = run["highly_variable_features"]

{
    "run_id": run.run_id,
    "active_cells": int(run.cells.fetch_all("I").sum()),
    "feature_selection": hvg_ref.artifact_id,
    "normalized": normalized.artifact_id,
}

{'run_id': 'a502c6ba45de187eb8aa73bca96bbd09fa2c5722976aaf534d2dce9207ffafab',
 'active_cells': 3940,
 'feature_selection': 'adeede36cdca822fde8cf2e62bb430843f28056a4e34414f753607c97a69240d',
 'normalized': '24cfc31e5c045f8066a38de7ce90d4ca1c6fe7d9c140e6b6686c77e731fd75d0'}

In [5]:
experimental = ExperimentalContextAgent(FunctionModel(_experimental_reply)).run(
    ds,
    study_context=(
        "Healthy-donor 5K PBMC. No treatment or batch labels are available. "
        "Do not invent a technical batch or biological contrast."
    ),
    run=run,
)

{
    "status": experimental.status,
    "batch_action": experimental.decision.batchCorrection.action,
    "batch_columns": experimental.decision.batchCorrection.batchColumns,
    "coefficients": experimental.decision.coefficientsOfInterest,
}

{'status': 'done',
 'batch_action': 'skip',
 'batch_columns': [],
 'coefficients': []}

In [6]:
if experimental.status != "done":
    raise RuntimeError(f"Experimental Context stopped with {experimental.status!r}")

tuning_handoff = experimental.to_parameter_tuning_handoff()
candidate = ParameterCandidate(
    candidateId="baseline",
    dimensions=15,
    leidenResolution=0.5,
    neighborsK=11,
    useHarmony=False,
)
tuning = ParameterTuningAgent(FunctionModel(_tuning_reply)).run(
    ds,
    normalized=normalized,
    candidates=[candidate],
    experimental_handoff=tuning_handoff,
    max_candidates=1,
    max_refined_candidates=0,
    min_cluster_cells=10,
)

evaluation = tuning.evaluations[0]
{
    "status": tuning.status,
    "recommended_candidate": tuning.recommendedCandidateId,
    "eligible": evaluation.eligible,
    "clusters": evaluation.metrics.nClusters,
    "smallest_cluster": evaluation.metrics.minClusterCells,
    "cluster_artifact": evaluation.artifacts["clusters"].artifactId,
    "cell_selection": evaluation.cellSelection.artifactId,
}

{'status': 'done',
 'recommended_candidate': 'baseline',
 'eligible': True,
 'clusters': 12,
 'smallest_cluster': 12,
 'cluster_artifact': 'c9ec9a51d0f0ea779459e20c1d030262aa50c6d766ad0e9b7dd02c09b423a195',
 'cell_selection': 'a4137c5ada6d3933e211d551dd636a097d6f0ed8103afde75fb3b2b05d67c16b'}

In [7]:
if tuning.status != "done":
    raise RuntimeError(f"Parameter Tuning stopped with {tuning.status!r}")

biology_handoff = tuning.to_biological_handoff()
biology = BiologicalInterpretationAgent(FunctionModel(_biology_reply)).run(
    ds,
    tuning_handoff=biology_handoff,
    biological_context=BiologicalContext(
        organism="Homo sapiens",
        tissue="peripheral blood",
        cellTypeReferences=["T cell", "B cell", "NK cell", "monocyte"],
        experimentalDetails=["healthy donor PBMC", "no treatment contrast"],
    ),
    allow_marker_search=True,
    marker_features=hvg_ref,
    max_clusters=1,
    max_markers=5,
    marker_min_score=0.01,
    marker_min_fraction=0.0,
)

{
    "status": biology.status,
    "interpretations": [
        {
            "cluster": item.clusterId,
            "identity": item.proposedIdentity,
            "rationale": item.rationale,
            "evidence": item.evidenceIds,
        }
        for item in biology.clusterInterpretations
    ],
    "tool_calls": [call.toolName for call in biology.runInfo.toolCalls],
    "treatment_observations": len(biology.treatmentObservations),
    "limitations": biology.limitations,
}

{'status': 'done',
 'interpretations': [{'cluster': '4',
   'identity': 'unresolved marker-defined cluster',
   'rationale': 'Top returned marker features: SOCS3, TCF7, TRAC.',
   'evidence': ['markers:4482f78c6a16d49ce8d19ec1de75cc13f76d43cce8b82d06509240d52937a762:clusters:c9ec9a51d0f0ea779459e20c1d030262aa50c6d766ad0e9b7dd02c09b423a195:cluster:4']}],
 'tool_calls': ['inspect_cluster_composition',
  'inspect_cluster_markers_batch'],
 'treatment_observations': 0,
 'limitations': ['The scripted documentation model does not assign cell identities.']}